# EP2: Agente Inteligente de Metro de Santiago
> **Ingenieria de Soluciones con IA** | *LangChain Agent con herramientas, memoria y planificacion*

**Indicadores de Logro:**
- **IL2.1:** Agente funcional con herramientas integradas (IE1, IE2)
- **IL2.2:** Memoria de corto y largo plazo (IE3, IE4)
- **IL2.3:** Planificacion y toma de decisiones (IE5, IE6)
- **IL2.4:** Documentacion tecnica del agente (IE7, IE8, IE9, IE10)

In [ ]:
import os
import json
import smtplib
import numpy as np
from datetime import datetime
from typing import List, Optional, Dict, Any
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

from dotenv import load_dotenv
from openai import OpenAI

from langchain.agents import AgentExecutor, create_openai_functions_agent
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.memory import ConversationBufferMemory
from langchain.memory.chat_memory import BaseChatMemory
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.schema import SystemMessage, HumanMessage, AIMessage
from langchain_core.messages import get_buffer_string
from pydantic import Field

load_dotenv()
GITHUB_BASE_URL = os.environ.get("GITHUB_BASE_URL")
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
print("Entorno configurado. API lista.")

---
## 1. Base de Conocimientos del Metro (IE4)
Contiene la informacion tecnica de tarifas, rutas y combinaciones del Metro de Santiago.
Se indexa con embeddings para busqueda semantica (recuperacion de contexto a largo plazo).

In [ ]:
class BaseConocimientoMetro:
    """Base de conocimientos con busqueda semantica via embeddings."""
    def __init__(self):
        self.client = OpenAI(base_url=GITHUB_BASE_URL, api_key=GITHUB_TOKEN)
        self.documentos = [
            "Tarifa PUNTA (07:00-08:59 y 18:00-19:59): Cuesta $840. Horario de alta demanda.",
            "Tarifa VALLE (09:00-17:59 y 20:00-20:44): Cuesta $760. Horario intermedio.",
            "Tarifa BAJO (06:00-06:59 y 20:45-23:00): Cuesta $680. Horario de baja demanda.",
            "TNE y Adulto Mayor: Tarifa especial de $240 en todo horario.",
            "Linea 1 (Roja): Terminales San Pablo - Los Dominicos. Conecta con L2, L3, L4, L5, L6.",
            "Linea 2 (Naranja): Terminales Vespucio Norte - La Cisterna.",
            "Linea 3 (Cafe): Terminales Plaza Quilicura - Fernando Castillo Velasco.",
            "Linea 4 (Azul): Terminales Tobalaba - Plaza Puente Alto. Conecta con L1, L3, L5.",
            "Linea 4A (Azul claro): Terminales La Cisterna - Vicuna Mackenna.",
            "Linea 5 (Verde): Terminales Plaza de Maipu - Vicente Valdes.",
            "Linea 6 (Morada): Terminales Cerrillos - Los Leones.",
            "COMBINACION: En Tobalaba puedes cambiar entre Linea 1 y Linea 4.",
            "COMBINACION: En Los Heroes puedes cambiar entre Linea 1 y Linea 2.",
            "COMBINACION: En Baquedano puedes cambiar entre Linea 1 y Linea 5.",
            "HORARIO SERVICIO: Lun-Vie 06:00-23:00. Sab 07:00-23:00. Dom 08:00-23:00.",
        ]
        self.embeddings_db = []
        self._indexar()

    def _get_embedding(self, texto: str) -> List[float]:
        response = self.client.embeddings.create(input=texto, model="text-embedding-3-small")
        return response.data[0].embedding

    def _indexar(self):
        print(f"Indexando {len(self.documentos)} documentos...")
        for doc in self.documentos:
            self.embeddings_db.append(self._get_embedding(doc))
        print("Indexacion completa.")

    def recuperar(self, consulta: str, top_k: int = 3) -> str:
        v_consulta = self._get_embedding(consulta)
        similitudes = [np.dot(v_consulta, v_db) for v_db in self.embeddings_db]
        indices = np.argsort(similitudes)[-top_k:][::-1]
        return "\n".join([self.documentos[i] for i in indices])

base_cm = BaseConocimientoMetro()
print("\nPrueba de recuperacion semantica:")
print(base_cm.recuperar("Cuanto cuesta viajar a las 8 de la manana?"))

---
## 2. Memoria Semantica a Largo Plazo (IE4)
Almacena interacciones previas como embeddings y recupera las mas relevantes
para mantener coherencia en flujos prolongados de conversacion.

In [ ]:
class MemoriaSemantica:
    """Memoria a largo plazo con recuperacion semantica.
    Cada interaccion se resume y almacena como embedding vectorial.
    Al recibir una nueva consulta, recupera las interacciones semanticamente
    similares para proveer contexto historico relevante."""

    def __init__(self):
        self.client = OpenAI(base_url=GITHUB_BASE_URL, api_key=GITHUB_TOKEN)
        self.interacciones: List[Dict[str, Any]] = []

    def _get_embedding(self, texto: str) -> List[float]:
        response = self.client.embeddings.create(input=texto, model="text-embedding-3-small")
        return response.data[0].embedding

    def guardar_interaccion(self, pregunta: str, respuesta: str) -> None:
        resumen = f"[Previa] Usuario: {pregunta[:150]} | Asistente: {respuesta[:150]}"
        embedding = self._get_embedding(resumen)
        self.interacciones.append({"resumen": resumen, "embedding": embedding, "timestamp": datetime.now().isoformat()})

    def recuperar_contexto(self, consulta: str, top_k: int = 2) -> str:
        if not self.interacciones:
            return ""
        v_consulta = self._get_embedding(consulta)
        similitudes = [np.dot(v_consulta, i["embedding"]) for i in self.interacciones]
        indices = np.argsort(similitudes)[-top_k:][::-1]
        return "\n".join([self.interacciones[i]["resumen"] for i in indices])

memoria_lp = MemoriaSemantica()
print("Memoria semantica lista.")

---
## 3. Herramientas del Agente (IE1)

Cinco herramientas que cubren las categorias requeridas:
- **Consulta:** `consultar_tarifa`, `consultar_ruta`
- **Escritura:** `generar_plan_viaje`, `enviar_correo`
- **Razonamiento:** `razonar_viaje`

In [ ]:
@tool
def consultar_tarifa(hora: str) -> str:
    """Consulta la tarifa del Metro de Santiago para una hora especifica.
    Args: hora en formato HH:MM (ej: 08:30, 18:00, 14:00)
    Returns: Informacion detallada de la tarifa aplicable en ese horario."""
    try:
        h, m = hora.split(":")
        minutos = int(h) * 60 + int(m)
    except:
        return "Formato de hora invalido. Use HH:MM (ej: 08:30)"
    if 420 <= minutos < 540 or 1080 <= minutos < 1200:
        return "Tarifa PUNTA ($840): Horario de alta demanda (07:00-08:59 y 18:00-19:59)."
    elif 540 <= minutos < 1080 or 1200 <= minutos < 1244:
        return "Tarifa VALLE ($760): Horario intermedio (09:00-17:59 y 20:00-20:44)."
    elif 360 <= minutos < 420 or 1244 <= minutos < 1380:
        return "Tarifa BAJO ($680): Horario de baja demanda (06:00-06:59 y 20:45-23:00)."
    else:
        return "El Metro no se encuentra en horario de servicio. Horario: Lun-Vie 06:00-23:00."

@tool
def consultar_ruta(origen: str, destino: str) -> str:
    """Consulta la ruta optima entre dos estaciones del Metro de Santiago.
    Args: origen, destino (nombre de estaciones)
    Returns: Descripcion de la ruta recomendada, lineas y combinaciones."""
    rutas = {
        ("plaza puente alto", "tobalaba"): "Linea 4 (Azul) directo sin combinaciones.",
        ("tobalaba", "plaza puente alto"): "Linea 4 (Azul) directo sin combinaciones.",
        ("plaza puente alto", "los dominicos"): "L4 hasta Tobalaba, combina con L1 hasta Los Dominicos.",
        ("los dominicos", "plaza puente alto"): "L1 hasta Tobalaba, combina con L4 hasta Plaza Puente Alto.",
        ("san pablo", "plaza puente alto"): "L1 hasta Tobalaba, combina con L4 hasta Plaza Puente Alto.",
        ("plaza puente alto", "san pablo"): "L4 hasta Tobalaba, combina con L1 hasta San Pablo.",
    }
    clave = (origen.strip().lower(), destino.strip().lower())
    if clave in rutas:
        return rutas[clave]
    return base_cm.recuperar(f"ruta desde {origen} hasta {destino}")

@tool
def generar_plan_viaje(origen: str, destino: str, hora: str) -> str:
    """Genera un plan de viaje completo y estructurado en formato JSON.
    Herramienta de ESCRITURA: produce un documento estructurado con todos
    los detalles del viaje: tarifa, ruta, combinaciones y recomendaciones.
    Args: origen, destino, hora en HH:MM
    Returns: String con JSON estructurado del plan de viaje."""
    tarifa = consultar_tarifa.invoke({"hora": hora})
    ruta = consultar_ruta.invoke({"origen": origen, "destino": destino})
    contexto_extra = base_cm.recuperar(f"viaje {origen} a {destino} a las {hora}")
    plan = {
        "tipo": "plan_viaje", "origen": origen, "destino": destino,
        "hora_consulta": hora, "tarifa": tarifa, "ruta": ruta,
        "fuentes_adicionales": contexto_extra,
        "generado_en": datetime.now().isoformat()
    }
    return json.dumps(plan, ensure_ascii=False, indent=2)

@tool
def razonar_viaje(pregunta: str) -> str:
    """Resuelve consultas complejas que requieren razonamiento multi-paso.
    Herramienta de RAZONAMIENTO: analiza preguntas que involucran multiples
    variables (tiempo, costo, rutas, restricciones) y produce respuesta integrada.
    Args: pregunta completa del usuario en lenguaje natural.
    Returns: Analisis detallado multi-paso con respuesta integrada."""
    contexto = base_cm.recuperar(pregunta, top_k=4)
    client = OpenAI(base_url=GITHUB_BASE_URL, api_key=GITHUB_TOKEN)
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": f"Eres analista de movilidad del Metro. Contexto:\n{contexto}\n\nEstructura tu respuesta en: 1) Horario y tarifa 2) Ruta optima 3) Costo 4) Recomendaciones"},
            {"role": "user", "content": pregunta}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content

@tool
def enviar_correo(destinatario: str, asunto: str, cuerpo: str) -> str:
    """Envia un correo electronico con los detalles de un plan de viaje.
    Herramienta de ESCRITURA: genera y envia automaticamente un reporte
    estructurado de viaje a un destinatario via correo electronico.
    Args: destinatario, asunto, cuerpo del mensaje.
    Returns: Confirmacion de envio o mensaje con ruta del archivo."""
    smtp_host = os.environ.get("SMTP_HOST", "smtp.gmail.com")
    smtp_port = int(os.environ.get("SMTP_PORT", "587"))
    smtp_user = os.environ.get("SMTP_USER", "")
    smtp_pass = os.environ.get("SMTP_PASS", "")
    remitente = os.environ.get("SMTP_FROM", smtp_user)
    if not smtp_user or not smtp_pass:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        fname = f"plan_viaje_{ts}.eml"
        with open(fname, "w", encoding="utf-8") as f:
            f.write(f"To: {destinatario}\nSubject: {asunto}\n\n{cuerpo}")
        return f"Correo guardado en {fname} (SMTP no configurado en .env)"
    try:
        msg = MIMEMultipart()
        msg["From"] = remitente
        msg["To"] = destinatario
        msg["Subject"] = asunto
        msg.attach(MIMEText(cuerpo, "plain", "utf-8"))
        with smtplib.SMTP(smtp_host, smtp_port) as server:
            server.starttls()
            server.login(smtp_user, smtp_pass)
            server.sendmail(remitente, destinatario, msg.as_string())
        return f"Correo enviado exitosamente a {destinatario}"
    except Exception as e:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        fname = f"plan_viaje_{ts}.eml"
        with open(fname, "w", encoding="utf-8") as f:
            f.write(f"To: {destinatario}\nSubject: {asunto}\n\n{cuerpo}")
        return f"Error SMTP: {str(e)}. Correo guardado en {fname}"

tools = [consultar_tarifa, consultar_ruta, generar_plan_viaje, razonar_viaje, enviar_correo]
print(f"5 herramientas configuradas: {[t.name for t in tools]}")

---
## 4. Memoria Compuesta: Corto + Largo Plazo (IE3 + IE4)
- **IE3 - Memoria de contenido:** `ConversationBufferMemory` mantiene las ultimas interacciones.
- **IE4 - Recuperacion semantica:** `MemoriaSemantica` recupera interacciones relevantes por similitud.
Ambas se combinan en una `MemoriaCompuesta` que el agente usa como memoria unificada.

In [ ]:
class MemoriaCompuesta(BaseChatMemory):
    """Memoria unificada: buffer conversacional (IE3) + recuperacion semantica (IE4).
    Provee chat_history (corto plazo) y semantic_context (largo plazo)."""
    chat_buffer: ConversationBufferMemory = Field(...)
    semantica: MemoriaSemantica = Field(...)

    @property
    def memory_variables(self) -> List[str]:
        return ["chat_history", "semantic_context"]

    def load_memory_variables(self, inputs: Dict[str, Any]) -> Dict[str, Any]:
        chat_vars = self.chat_buffer.load_memory_variables(inputs)
        user_input = inputs.get("input", "")
        contexto = self.semantica.recuperar_contexto(user_input)
        return {
            "chat_history": chat_vars.get("chat_history", []),
            "semantic_context": contexto if contexto else "No hay interacciones previas relevantes."
        }

    def save_context(self, inputs: Dict[str, Any], outputs: Dict[str, Any]) -> None:
        self.chat_buffer.save_context(inputs, outputs)
        self.semantica.guardar_interaccion(inputs.get("input", ""), outputs.get("output", ""))

    def clear(self) -> None:
        self.chat_buffer.clear()
        self.semantica.interacciones.clear()

memoria_agente = MemoriaCompuesta(
    chat_buffer=ConversationBufferMemory(memory_key="chat_history", return_messages=True, max_token_limit=3000),
    semantica=memoria_lp
)
print("Memoria compuesta configurada (corto + largo plazo).")

---
## 5. Creacion del Agente con Planificacion (IE2 + IE5)
**Framework:** LangChain Agent con GPT-4o via GitHub Models API.
**Planificacion (IE5):** ReAct (Razonamiento + Accion). El prompt instruye al agente a descomponer problemas complejos.
**Toma de decisiones (IE6):** Selecciona herramientas segun el tipo de consulta y adapta recomendaciones segun horario.

In [ ]:
llm = ChatOpenAI(base_url=GITHUB_BASE_URL, api_key=GITHUB_TOKEN, model="gpt-4o", temperature=0.3)

SISTEMA_PROMPT = """
Eres un agente de movilidad del Metro de Santiago.

## PLANIFICACION (IE5)
Sigue estos pasos:
1. ANALIZA: Identifica que necesita el usuario
2. PLANIFICA: Decide que herramientas usar y en que orden
3. EJECUTA: Usa las herramientas una por una
4. INTEGRA: Combina resultados en una respuesta clara
5. VERIFICA: Confirma que la respuesta cubre toda la consulta

## TOMA DE DECISIONES (IE6)
- Solo tarifa -> consultar_tarifa
- Solo ruta -> consultar_ruta
- Viaje completo -> generar_plan_viaje
- Consulta compleja o multiple -> razonar_viaje
- Enviar por correo -> enviar_correo
- Ambigua -> pide aclaracion

NO improvises tarifas o rutas; siempre consulta las herramientas.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SISTEMA_PROMPT),
    ("system", "Contexto de interacciones previas relevantes:\n{semantic_context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_openai_functions_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent, tools=tools, memory=memoria_agente,
    verbose=True, handle_parsing_errors=True,
    max_iterations=10, early_stopping_method="force"
)

def ejecutar_agente(pregunta: str) -> str:
    return agent_executor.invoke({"input": pregunta})["output"]

print("Agente LangChain creado con planificacion (ReAct).")

---
## 6. Demostraciones del Agente (IE6)
Cada demo muestra un aspecto diferente de la toma de decisiones del agente.

### Demo 1: Consulta de Tarifa (Consulta)
El agente identifica que solo necesita `consultar_tarifa`.

In [ ]:
print("=" * 60)
print("DEMO 1: Consulta simple de tarifa")
print("=" * 60)
r = ejecutar_agente("Cuanto cuesta el pasaje a las 8:30 de la manana?")
print(f"Respuesta: {r}")

### Demo 2: Plan de Viaje Estructurado (Escritura)
El agente usa `generar_plan_viaje` para producir JSON.

In [ ]:
print("=" * 60)
print("DEMO 2: Plan de viaje completo (escritura)")
print("=" * 60)
r = ejecutar_agente("Necesito un plan de viaje desde Plaza Puente Alto hasta Los Dominicos a las 18:30")
print(f"Respuesta:\n{r}")

### Demo 3: Razonamiento Multi-Paso (Razonamiento)
El agente usa `razonar_viaje` para analizar consultas que integran tarifa, ruta y horario.

In [ ]:
print("=" * 60)
print("DEMO 3: Razonamiento multi-paso")
print("=" * 60)
r = ejecutar_agente(
    "Voy desde San Pablo hasta Plaza Puente Alto a las 19:30. "
    "Quiero saber cuanto me cuesta, que ruta tomar, y si hay alguna "
    "alternativa mas economica si viajo mas tarde."
)
print(f"Respuesta:\n{r}")

### Demo 4: Conversacion Multi-Turno con Memoria (IE3)
El agente recuerda la conversacion anterior y mantiene coherencia.

In [ ]:
print("=" * 60)
print("DEMO 4: Memoria de corto plazo")
print("=" * 60)
r1 = ejecutar_agente("Cual es la tarifa para viajar a las 8:00?")
print(f"[Turno 1] R: {r1}\n")
r2 = ejecutar_agente("Y si viajo dos horas despues, cuanto pagaria?")
print(f"[Turno 2] R: {r2}")

### Demo 5: Memoria Semantica a Largo Plazo (IE4)
Recupera interacciones previas semanticamente similares.

In [ ]:
print("=" * 60)
print("DEMO 5: Memoria semantica a largo plazo")
print("=" * 60)
print("Guardando interaccion previa en memoria semantica...")
memoria_agente.semantica.guardar_interaccion(
    "Cual es la tarifa para la tercera edad?",
    "La tarifa para adultos mayores es de $240 en todo horario."
)
print("Listo. Preguntando algo relacionado...\n")
r = ejecutar_agente("Mi abuela quiere viajar, cuanto paga?")
print(f"Respuesta: {r}")

### Demo 6: Envio Automatico de Plan por Correo (Escritura)
El agente usa `enviar_correo` para enviar el plan de viaje.
Si SMTP no esta configurado en .env, guarda el correo como archivo .eml local.

In [ ]:
print("=" * 60)
print("DEMO 6: Envio automatico de plan por correo")
print("=" * 60)
r = ejecutar_agente("Envia por correo a juan@ejemplo.cl el plan de viaje desde San Pablo hasta Puente Alto a las 7:45")
print(f"Respuesta:\n{r}")

---
## 7. Analisis de Decisiones del Agente (IE6)
Documenta como el agente toma decisiones segun diferentes escenarios.

In [ ]:
escenarios = [
    {"nombre": "A: Consulta tarifa", "input": "Cuanto vale viajar a las 14:30?",
     "decision": "consultar_tarifa (consulta directa)"},
    {"nombre": "B: Consulta ruta", "input": "Como llego de Tobalaba a San Pablo?",
     "decision": "consultar_ruta (consulta directa)"},
    {"nombre": "C: Plan completo", "input": "Planifica viaje Los Dominicos a Puente Alto a las 20:00",
     "decision": "generar_plan_viaje (escritura)"},
    {"nombre": "D: Consulta compleja", "input": "Que me conviene si viajo a las 7:45 o espero a las 9:00?",
     "decision": "razonar_viaje (razonamiento multi-paso)"},
    {"nombre": "E: Correo automatico", "input": "Envia a juan@correo.cl el plan para viajar de San Pablo a Puente Alto a las 8:00",
     "decision": "enviar_correo (escritura + envio)"},
    {"nombre": "F: Seguimiento memoria", "input": "Y si voy media hora antes?",
     "decision": "chat_history + herramienta adecuada (memoria IE3)"},
]

print("Tabla de decisiones del agente:")
print("-"*80)
print(f"{'Escenario':<30} {'Decision':<50}")
print("-"*80)
for e in escenarios:
    print(f"{e['nombre']:<30} {e['decision']:<50}")
print("-"*80)